# 04 - Market Operations Stress

## Purpose

Create a day-region stress output combining market pressure and operational incident pressure.

This notebook identifies days and regions where market conditions and grid operations both show stress signals.

## Expected output

`vattenfall_dev.analytics.gold_market_operations_stress`

## Main metrics

- average market price
- maximum market price
- total market volume
- incident count
- elevated incident count
- total incident duration
- market pressure flag
- operations stress flag
- combined stress label

## Main idea

This output helps identify combined market and operations pressure in a simple, explainable way.

In [0]:
import sys
import importlib
from pyspark.sql import functions as F

repo_src_path = "/Workspace/Users/dirella@startsteps.org/vattenfall-week9-capstone-anna/src"

if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

import risk.risk_classification as risk_classification
importlib.reload(risk_classification)

from risk.risk_classification import add_combined_stress_label

In [0]:
catalog = "vattenfall_dev"

market_table = f"{catalog}.analytics.gold_daily_market_summary"
grid_table = f"{catalog}.analytics.gold_grid_incident_summary"

target_table = f"{catalog}.analytics.gold_market_operations_stress"

print("Market source:", market_table)
print("Grid source:", grid_table)
print("Target table:", target_table)

In [0]:
market_df = spark.table(market_table)
grid_df = spark.table(grid_table)

print("Market rows:", market_df.count())
print("Grid incident rows:", grid_df.count())

display(market_df)
display(grid_df)

In [0]:
grid_day_region_df = (
    grid_df
    .groupBy(
        F.col("event_day").alias("report_day"),
        "region"
    )
    .agg(
        F.sum("incident_count").alias("incident_count"),
        F.sum("elevated_incident_count").alias("elevated_incident_count"),
        F.sum("total_duration_minutes").alias("total_duration_minutes"),
    )
)

display(grid_day_region_df)

In [0]:
market_ops_df = (
    market_df.alias("m")
    .join(
        grid_day_region_df.alias("g"),
        on=["report_day", "region"],
        how="left"
    )
    .fillna(
        {
            "incident_count": 0,
            "elevated_incident_count": 0,
            "total_duration_minutes": 0,
        }
    )
    .withColumn(
        "market_pressure_flag",
        F.when(F.col("is_high_market_price") == 1, 1).otherwise(0)
    )
    .withColumn(
        "operations_stress_flag",
        F.when(
            (F.col("elevated_incident_count") > 0)
            | (F.col("total_duration_minutes") >= 60),
            1
        ).otherwise(0)
    )
    .transform(add_combined_stress_label)
)

display(market_ops_df)

In [0]:
(
    market_ops_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print("Written Night Shift market-operations stress table:", target_table)
print("Rows written:", market_ops_df.count())

In [0]:
result_df = spark.table(target_table)

print("Rows in market-operations stress:", result_df.count())
print("Columns:", result_df.columns)

display(result_df)

print("Combined stress label distribution:")
result_df.groupBy("combined_stress_label").count().show()

print("Market pressure flag distribution:")
result_df.groupBy("market_pressure_flag").count().show()

print("Operations stress flag distribution:")
result_df.groupBy("operations_stress_flag").count().show()

In [0]:
duplicate_rows = (
    result_df
    .groupBy("report_day", "region")
    .count()
    .filter("count > 1")
    .count()
)

print("Duplicate report_day-region rows:", duplicate_rows)

if duplicate_rows > 0:
    raise ValueError("Market-operations stress validation failed: duplicate report_day-region rows found.")

print("Market-operations stress validation passed.")